# c. Crawling Berita

In [ ]:
!pip install requests
!pip install beautifulsoup4
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
!pip install builtwith

  Preparing metadata (setup.py) ... done
  Created wheel for builtwith: filename=builtwith-1.3.4-py3-none-any.whl size=36077 sha256=bbf93d04d625180933c22fd6c25e231a71f66995b5daff32fdacba6907a3addc
  Stored in directory: /root/.cache/pip/wheels/7f/2d/b2/606e3df914d4aeeab99c4a4e3e9a61673d2293c2e346db00c8
Successfully built builtwith


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urlparse

def scrape_kompas_article(url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        r = requests.get(url, headers=headers)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error saat mengambil URL: {e}")
        return None

    soup = BeautifulSoup(r.content, "html.parser")

    # Judul
    judul = soup.select_one("h1.read__title")
    judul_text = judul.text.strip() if judul else "Tidak ditemukan judul"

    # Isi (100 kata)
    isi_elems = soup.select("div.read__content p")
    isi_text = " ".join([p.get_text(strip=True) for p in isi_elems]) if isi_elems else "Tidak ditemukan isi"
    words = isi_text.split()
    isi_100 = " ".join(words[:100]) + ("..." if len(words) > 100 else "")

    # Cari kategori
    kategori_text = "Tidak ditemukan kategori"

    kategori_meta = soup.find("meta", {"property": "article:section"})
    if kategori_meta and kategori_meta.get("content"):
        kategori_text = kategori_meta["content"].strip()

    if kategori_text == "Tidak ditemukan kategori":
        breadcrumb = soup.select("div.breadcrumb__link a")
        if breadcrumb and len(breadcrumb) > 1:
            kategori_text = breadcrumb[1].get_text(strip=True)

    if kategori_text == "Tidak ditemukan kategori":
        parsed = urlparse(url)
        subdomain = parsed.netloc.split(".")[0]
        if subdomain and subdomain != "www" and subdomain != "kompas":
            kategori_text = subdomain.capitalize()

    return {
        "Judul": judul_text,
        "Isi (100 kata)": isi_100,
        "Kategori": kategori_text
    }

# Daftar URL
urls_to_test = [
    "https://lifestyle.kompas.com/read/2025/08/31/100000720/waspadai-kelelahan-mental-akibat-kebanyakan-berita-negatif-",
    "https://money.kompas.com/read/2025/08/18/134653726/pendidikan-kewirausahaan-yang-merdeka",
    "https://health.kompas.com/read/25H19130000068/dokter--olahraga-bisa-turunkan-risiko-kanker-asal-rutin-dan-benar",
    "https://nasional.kompas.com/read/2025/09/03/18022971/peristiwa-gas-air-mata-unisba-mendikti-janjikan-pendampingan-dan",
    "https://nasional.kompas.com/read/2025/08/19/07251961/harapan-dan-catatan-soal-anggaran-pendidikan-terbesar-sepanjang-sejarah-ri",
    "https://travel.kompas.com/read/2025/09/04/210100827/berdarah-belanda-depok-pesepak-bola-miliano-jonathans-resmi-jadi-wni"
]

results = [scrape_kompas_article(u) for u in urls_to_test if scrape_kompas_article(u)]
df = pd.DataFrame(results)

from IPython.display import display
display(df)

,Judul,Isi (100 kata),Kategori
0,Waspadai Kelelahan Mental akibat Kebanyakan Be...,KOMPAS.com -Berbagai informasi peristiwa terba...,Lifestyle
1,Pendidikan Kewirausahaan yang Merdeka,"SETIAP17 Agustus, Masyarakat Indonesia merayak...",Money
2,"Dokter: Olahraga Bisa Turunkan Risiko Kanker, ...",KOMPAS.com –Health Management Specialist Corpo...,Health
3,"Peristiwa Gas Air Mata Unisba, Mendikti Janjik...","JAKARTA, KOMPAS.com- Menteri Pendidikan Tinggi...",Nasional
4,Harapan dan Catatan soal Anggaran Pendidikan T...,"JAKARTA, KOMPAS.com- Presiden Prabowo Subianto...",Nasional
5,"Berdarah Belanda Depok, Pesepak Bola Miliano J...",KOMPAS.com -Kementerian Hukum Republik Indones...,Travel
